# Proof Step Graph Analysis

Load traced proof step graphs from JSONL and compute statistics / distributions.

In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from proof_step_graph.graph import ProofStepGraph, NODE_GOAL, NODE_TACTIC

In [ ]:
# ── Load graphs ──────────────────────────────────────────────────────────────
JSONL_PATH = Path("../data/YOUR_GRAPHS.jsonl")  # <-- change this

graphs = []
with JSONL_PATH.open() as f:
    for line in f:
        line = line.strip()
        if line:
            graphs.append(ProofStepGraph.from_dict(json.loads(line)))

print(f"Loaded {len(graphs)} proof step graphs")

## Summary statistics

In [ ]:
df = pd.DataFrame([pg.stats() for pg in graphs])

print(f"{'='*55}")
print(f"  Proof Step Graph Summary  ({len(df)} theorems)")
print(f"{'='*55}")
for col in df.select_dtypes(include="number").columns:
    print(f"  {col:<25} mean={df[col].mean():.2f}  max={df[col].max()}")
print(f"  is_dag ratio: {df['is_dag'].mean():.2%}")

df.describe()

## Distributions

In [ ]:
cols = ["n_goals", "n_tactics", "n_initial_goals", "max_branching", "avg_branching"]

fig, axes = plt.subplots(1, len(cols), figsize=(4 * len(cols), 4))
for ax, col in zip(axes, cols):
    ax.hist(df[col].dropna(), bins=20, edgecolor="black")
    ax.set_title(col)
    ax.set_xlabel("value")
    ax.set_ylabel("count")
fig.tight_layout()
plt.show()

## Branching vs depth

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(df["n_tactics"], df["max_branching"], alpha=0.5, s=20)
ax.set_xlabel("proof depth (# tactics)")
ax.set_ylabel("max branching factor")
ax.set_title("Branching vs Depth")
fig.tight_layout()
plt.show()

## Tactic frequency

In [ ]:
TOP_N = 20

counter: Counter = Counter()
for pg in graphs:
    for _, d in pg.tactic_nodes():
        tac = d["tactic"].split()[0].rstrip(";").rstrip("<;>")
        counter[tac] += 1

top = counter.most_common(TOP_N)
names, counts = zip(*top) if top else ([], [])

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(names, counts, edgecolor="black")
ax.set_xlabel("frequency")
ax.set_title(f"Top-{TOP_N} tactic names")
ax.invert_yaxis()
fig.tight_layout()
plt.show()